In [1]:
# 2. データ確認セル
import os
import json
import numpy as np
import pandas as pd
from pathlib import Path

# /kaggle/input/competitions/playground-series-s6e5/train.csv
BASE = Path("/kaggle/input/competitions/playground-series-s6e5")

train_path = BASE / "train.csv"
test_path = BASE / "test.csv"
sub_path = BASE / "sample_submission.csv"

train = pd.read_csv(train_path)
test = pd.read_csv(test_path)
sub = pd.read_csv(sub_path)

print("train shape:", train.shape)
print("test shape :", test.shape)
print("sub shape  :", sub.shape)

print("\ntrain columns:")
print(train.columns.tolist())

print("\ntest columns:")
print(test.columns.tolist())

print("\nsample_submission columns:")
print(sub.columns.tolist())

# id列推定
id_col = "id" if "id" in train.columns else sub.columns[0]

# target列推定: trainにありtestにない列からidを除外
target_candidates = [c for c in train.columns if c not in test.columns and c != id_col]
print("\nid_col:", id_col)
print("target_candidates:", target_candidates)

assert len(target_candidates) == 1, f"target候補が1つに定まりません: {target_candidates}"
target_col = target_candidates[0]
print("target_col:", target_col)

# train/test feature一致確認
feature_cols = [c for c in test.columns if c != id_col]
train_feature_cols = [c for c in train.columns if c not in [id_col, target_col]]

print("\nfeature cols match:", feature_cols == train_feature_cols)

missing_report = pd.DataFrame({
    "column": train.columns,
    "train_dtype": [str(train[c].dtype) for c in train.columns],
    "train_missing": [train[c].isna().sum() for c in train.columns],
    "train_missing_rate": [train[c].isna().mean() for c in train.columns],
    "train_nunique": [train[c].nunique(dropna=False) for c in train.columns],
})

test_missing_report = pd.DataFrame({
    "column": test.columns,
    "test_dtype": [str(test[c].dtype) for c in test.columns],
    "test_missing": [test[c].isna().sum() for c in test.columns],
    "test_missing_rate": [test[c].isna().mean() for c in test.columns],
    "test_nunique": [test[c].nunique(dropna=False) for c in test.columns],
})

display(missing_report)
display(test_missing_report)

print("\ntarget distribution:")
print(train[target_col].value_counts(dropna=False))
print("\ntarget ratio:")
print(train[target_col].value_counts(normalize=True, dropna=False))

train shape: (439140, 16)
test shape : (188165, 15)
sub shape  : (188165, 2)

train columns:
['id', 'Driver', 'Compound', 'Race', 'Year', 'PitStop', 'LapNumber', 'Stint', 'TyreLife', 'Position', 'LapTime (s)', 'LapTime_Delta', 'Cumulative_Degradation', 'RaceProgress', 'Position_Change', 'PitNextLap']

test columns:
['id', 'Driver', 'Compound', 'Race', 'Year', 'PitStop', 'LapNumber', 'Stint', 'TyreLife', 'Position', 'LapTime (s)', 'LapTime_Delta', 'Cumulative_Degradation', 'RaceProgress', 'Position_Change']

sample_submission columns:
['id', 'PitNextLap']

id_col: id
target_candidates: ['PitNextLap']
target_col: PitNextLap

feature cols match: True


,column,train_dtype,train_missing,train_missing_rate,train_nunique
0,id,int64,0,0.0,439140
1,Driver,object,0,0.0,887
2,Compound,object,0,0.0,5
3,Race,object,0,0.0,26
4,Year,int64,0,0.0,4
5,PitStop,int64,0,0.0,2
6,LapNumber,int64,0,0.0,78
7,Stint,int64,0,0.0,8
8,TyreLife,float64,0,0.0,78
9,Position,int64,0,0.0,20


,column,test_dtype,test_missing,test_missing_rate,test_nunique
0,id,int64,0,0.0,188165
1,Driver,object,0,0.0,801
2,Compound,object,0,0.0,5
3,Race,object,0,0.0,26
4,Year,int64,0,0.0,4
5,PitStop,int64,0,0.0,2
6,LapNumber,int64,0,0.0,77
7,Stint,int64,0,0.0,8
8,TyreLife,float64,0,0.0,77
9,Position,int64,0,0.0,20



target distribution:
PitNextLap
0.0    351759
1.0     87381
Name: count, dtype: int64

target ratio:
PitNextLap
0.0    0.801018
1.0    0.198982
Name: proportion, dtype: float64


In [2]:
# 3. 列グループ整理セル
import re

def infer_column_groups(df, id_col, target_col=None):
    ignore = {id_col}
    if target_col is not None:
        ignore.add(target_col)

    cols = [c for c in df.columns if c not in ignore]

    race_pat = re.compile(r"(race|circuit|grand|prix|round|event|season|year)", re.I)
    driver_pat = re.compile(r"(driver|constructor|team)", re.I)
    lap_pat = re.compile(r"(lap|stint|pit|stop)", re.I)
    tyre_pat = re.compile(r"(tyre|tire|compound|fresh|age)", re.I)
    time_pat = re.compile(r"(time|duration|second|sec|millisecond|ms|date)", re.I)

    groups = {
        "race_like": [],
        "driver_team_like": [],
        "lap_stint_pit_like": [],
        "tyre_like": [],
        "time_like": [],
        "categorical": [],
        "numerical": [],
        "bool_like": [],
    }

    n = len(df)

    for c in cols:
        s = df[c]
        nunique = s.nunique(dropna=False)
        dtype = s.dtype

        if race_pat.search(c):
            groups["race_like"].append(c)
        if driver_pat.search(c):
            groups["driver_team_like"].append(c)
        if lap_pat.search(c):
            groups["lap_stint_pit_like"].append(c)
        if tyre_pat.search(c):
            groups["tyre_like"].append(c)
        if time_pat.search(c):
            groups["time_like"].append(c)

        if s.dropna().isin([0, 1, True, False]).all() and nunique <= 3:
            groups["bool_like"].append(c)

        if dtype == "object" or str(dtype) == "category" or dtype == "bool":
            groups["categorical"].append(c)
        elif pd.api.types.is_integer_dtype(dtype) and nunique <= min(100, max(20, int(n * 0.01))):
            groups["categorical"].append(c)
        elif pd.api.types.is_numeric_dtype(dtype):
            groups["numerical"].append(c)
        else:
            groups["categorical"].append(c)

    # 重複除去
    for k in groups:
        groups[k] = sorted(list(dict.fromkeys(groups[k])))

    return groups

groups = infer_column_groups(train, id_col=id_col, target_col=target_col)

for k, v in groups.items():
    print(f"\n[{k}] {len(v)}")
    print(v)

with open("s6e5_column_groups.json", "w", encoding="utf-8") as f:
    json.dump(groups, f, ensure_ascii=False, indent=2)


[race_like] 3
['Race', 'RaceProgress', 'Year']

[driver_team_like] 1
['Driver']

[lap_stint_pit_like] 5
['LapNumber', 'LapTime (s)', 'LapTime_Delta', 'PitStop', 'Stint']

[tyre_like] 2
['Compound', 'TyreLife']

[time_like] 2
['LapTime (s)', 'LapTime_Delta']

[categorical] 8
['Compound', 'Driver', 'LapNumber', 'PitStop', 'Position', 'Race', 'Stint', 'Year']

[numerical] 6
['Cumulative_Degradation', 'LapTime (s)', 'LapTime_Delta', 'Position_Change', 'RaceProgress', 'TyreLife']

[bool_like] 1
['PitStop']


In [3]:
from catboost import CatBoostClassifier, Pool
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score
import numpy as np
import pandas as pd
import json
from pathlib import Path

SEED = 42
N_SPLITS = 5
EXP_NAME = "exp_20260501_002_cat_strict_raw_baseline"
OUTDIR = Path(f"/kaggle/working/{EXP_NAME}")
OUTDIR.mkdir(parents=True, exist_ok=True)

X = train[train_feature_cols].copy()
X_test = test[train_feature_cols].copy()
y = train[target_col].copy()

if y.dtype == "object" or str(y.dtype) == "category":
    uniques = sorted(y.dropna().unique().tolist())
    assert len(uniques) == 2, f"AUC前提だがtargetが2値ではありません: {uniques}"
    target_map = {v: i for i, v in enumerate(uniques)}
    y = y.map(target_map).astype(int)
else:
    target_map = None
    y = y.astype(int)

cat_cols = groups["categorical"]
cat_cols = [c for c in cat_cols if c in train_feature_cols]

# CatBoostは文字列カテゴリの欠損を明示的に埋める
for c in cat_cols:
    X[c] = X[c].astype("string").fillna("__MISSING__")
    X_test[c] = X_test[c].astype("string").fillna("__MISSING__")

cat_features = [X.columns.get_loc(c) for c in cat_cols]

oof = np.zeros(len(train))
pred = np.zeros(len(test))
fold_scores = []

skf = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=SEED)

params = {
    "loss_function": "Logloss",
    "eval_metric": "AUC",
    "iterations": 5000,
    "learning_rate": 0.03,
    "depth": 6,
    "l2_leaf_reg": 3.0,
    "random_seed": SEED,
    "early_stopping_rounds": 300,
    "verbose": 300,
    "allow_writing_files": False,
}

for fold, (tr_idx, va_idx) in enumerate(skf.split(X, y), 1):
    X_tr, X_va = X.iloc[tr_idx], X.iloc[va_idx]
    y_tr, y_va = y.iloc[tr_idx], y.iloc[va_idx]

    train_pool = Pool(X_tr, y_tr, cat_features=cat_features)
    valid_pool = Pool(X_va, y_va, cat_features=cat_features)
    test_pool = Pool(X_test, cat_features=cat_features)

    model = CatBoostClassifier(**params)
    model.fit(train_pool, eval_set=valid_pool, use_best_model=True)

    va_pred = model.predict_proba(valid_pool)[:, 1]
    te_pred = model.predict_proba(test_pool)[:, 1]

    oof[va_idx] = va_pred
    pred += te_pred / N_SPLITS

    auc = roc_auc_score(y_va, va_pred)
    fold_scores.append(auc)
    print(f"fold {fold}: AUC={auc:.6f}, best_iter={model.get_best_iteration()}")

cv = roc_auc_score(y, oof)
print("CV AUC:", cv)
print("fold scores:", fold_scores)
print("mean fold:", np.mean(fold_scores), "std:", np.std(fold_scores))

np.save(OUTDIR / f"oof_{EXP_NAME}.npy", oof)
np.save(OUTDIR / f"pred_{EXP_NAME}.npy", pred)

sub_out = sub.copy()
pred_col = [c for c in sub_out.columns if c != id_col][0]
sub_out[pred_col] = pred
sub_out.to_csv(OUTDIR / f"submission_{EXP_NAME}.csv", index=False)

result = {
    "exp_name": EXP_NAME,
    "model": "CatBoost",
    "cv_auc": float(cv),
    "fold_scores": [float(x) for x in fold_scores],
    "target_col": target_col,
    "id_col": id_col,
    "target_map": target_map,
    "n_train": int(len(train)),
    "n_test": int(len(test)),
    "cat_cols": cat_cols,
    "params": params,
}

with open(OUTDIR / "cv_result.json", "w", encoding="utf-8") as f:
    json.dump(result, f, ensure_ascii=False, indent=2)

pd.DataFrame({"feature": train_feature_cols}).to_csv(OUTDIR / "feature_columns.csv", index=False)

print("saved to:", OUTDIR)
print("submission:", OUTDIR / f"submission_{EXP_NAME}.csv")

0:	test: 0.9010507	best: 0.9010507 (0)	total: 538ms	remaining: 44m 50s
300:	test: 0.9386671	best: 0.9386671 (300)	total: 1m 55s	remaining: 30m 4s
600:	test: 0.9430757	best: 0.9430757 (600)	total: 3m 55s	remaining: 28m 44s
900:	test: 0.9450767	best: 0.9450767 (900)	total: 5m 57s	remaining: 27m 4s
1200:	test: 0.9462445	best: 0.9462445 (1200)	total: 7m 59s	remaining: 25m 17s
1500:	test: 0.9469548	best: 0.9469548 (1500)	total: 10m 1s	remaining: 23m 22s
1800:	test: 0.9475459	best: 0.9475459 (1800)	total: 12m 4s	remaining: 21m 26s
2100:	test: 0.9479339	best: 0.9479342 (2099)	total: 14m 7s	remaining: 19m 28s
2400:	test: 0.9482551	best: 0.9482551 (2400)	total: 16m 10s	remaining: 17m 30s
2700:	test: 0.9484979	best: 0.9484979 (2700)	total: 18m 15s	remaining: 15m 32s
3000:	test: 0.9487077	best: 0.9487083 (2999)	total: 20m 20s	remaining: 13m 33s
3300:	test: 0.9488878	best: 0.9488878 (3300)	total: 22m 26s	remaining: 11m 33s
3600:	test: 0.9490439	best: 0.9490441 (3598)	total: 24m 32s	remaining: 9m 3